# Lendo o arquivo bruto

In [0]:
#Lendo o arquivo json que está no datalake
custos_importacao = spark.read.option("multiline", "true").json("/Volumes/lh_nautical/datalake/datas/custos_importacao.json")
custos_importacao.display()


# Salvando o arquivo bruto como tabela delta na camada Bronze

In [0]:
%sql
--Informando o catalogo e schema
USE CATALOG lh_nautical;
USE SCHEMA bronze_lh_nautical

In [0]:
#criando a tabela na camada bronze
custos_importacao.write.mode("overwrite").format("delta").saveAsTable("bronze_lh_nautical.tbl_custos_importacao")

# Conferindo a tabela delta criada na camada Bronze

In [0]:
#Conferindo a tabela
tbl_custos_importacao = spark.read.table("bronze_lh_nautical.tbl_custos_importacao")
tbl_custos_importacao.display()

# Recriando a tabela para camada Silver

### - Separando a lista historic_data

In [0]:
from pyspark.sql.functions import explode,col
#Separando a coluna historic_data em duas colunas para facilitar a manipulação
tbl_custos_importacao_clean = tbl_custos_importacao.select('*',explode(col('historic_data'))).select('*',col('col.*'))
#excluindo as colunas que não serão utilizadas
tbl_custos_importacao_clean = tbl_custos_importacao_clean.drop('col').drop('historic_data')
#criando a tabela na camada silver
tbl_custos_importacao_clean.write.mode("overwrite").format("delta").saveAsTable("silver_lh_nautical.tbl_custos_importacao")
#Conferindo a tabela
tbl_custos_importacao_clean = spark.read.table("silver_lh_nautical.tbl_custos_importacao")
tbl_custos_importacao_clean.display()

### - Mudando os tipos das coluna

In [0]:
from pyspark.sql.types import *
from pyspark.sql.functions import when, col
# Modificando os tipos de dados
tbl_custos_importacao_clean.withColumn('product_id', col('product_id').cast(IntegerType())).withColumn('usd_price', col('usd_price').cast(DecimalType(10,2))).write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("silver_lh_nautical.tbl_custos_importacao")
tbl_custos_importacao_clean.display()

### - Formatando as datas e modificando o tipo da coluna start_date


In [0]:
from pyspark.sql.functions import regexp_replace, to_date
from pyspark.sql import functions as F
# Modificando todos os caracteres / para - na coluna start_date
tbl_custos_importacao_clean = spark.read.table("silver_lh_nautical.tbl_custos_importacao")
tbl_custos_importacao_clean = tbl_custos_importacao_clean.withColumn("start_date", regexp_replace(col("start_date"), "/", "-"))
# Salvando a modificação
tbl_custos_importacao_clean.write.mode("overwrite").format("delta").saveAsTable("silver_lh_nautical.tbl_custos_importacao")
tbl_custos_importacao_clean.display()
# Alterando o tipo de dados da coluna start_date
tbl_custos_importacao_clean = tbl_custos_importacao_clean.withColumn("start_date", F.date_format(F.to_date(col("start_date"), "dd-MM-yyyy"), "yyyy-MM-dd").cast("date"))
#Salvando a modificação
tbl_custos_importacao_clean.display()
tbl_custos_importacao_clean.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("silver_lh_nautical.tbl_custos_importacao")

### - Formatando a coluna category

In [0]:

tbl_custos_importacao_clean = spark.read.table("silver_lh_nautical.tbl_custos_importacao")

# Padronizando os dados da coluna category
tbl_custos_importacao_clean = tbl_custos_importacao_clean.withColumn("category", when(tbl_custos_importacao_clean.category.startswith("ele"), "Eletrônicos").when(tbl_custos_importacao_clean.category.startswith("prop"), "Propulsão").when(tbl_custos_importacao_clean.category.startswith("anc"), "Ancoragem"))
#Salvando a modificação
tbl_custos_importacao_clean.display()
tbl_custos_importacao_clean.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("silver_lh_nautical.tbl_custos_importacao")
